## Knowledge Graph Construction

This notebook covers the development of a clinical knowledge graph created from the synthetic patient records generated. First, an RDF/Turtle knowledge graph is constructed, creating the following nodes: patient (with unique identifier and opioid toxicity risk level), symptom, vital sign and risk factor. Each node is defined using established ontologies such as Schema.org or SNOMED.CT. The second part of the notebook includes the creation of the network knowledge graph, used as the base for an interactive visualization of the graph to better understand how various nodes are connected. The interactive graph can be found at: https://lite.gephi.org/?file=https://gist.githubusercontent.com/mluisab/a48178b9ed4db858a2bf1c46e03c15c9/raw/870e83dfb02ed8b38000e6057bbf911e424ce729/knowledge_graph.graphml.json 

In [1]:
import pandas as pd
import json
from rdflib import Graph, Literal, RDF, Namespace
from rdflib.namespace import RDFS, RDF, OWL
import networkx as nx

In [2]:
#Import and read dataset
df = pd.read_csv("MV_dataset.csv")

#Import and read JSON specification file
with open("feature_spec.json", "r") as file:
    spec = json.load(file)

In [3]:
#Gather all features from specification file 
feature_map = {}

for symptom in spec["symptoms"]:
    feature_map[symptom["SymptomID"]] = {
        "category": "symptom",
        "fsn": symptom["FSN"],
        "sctid": symptom["SCTID"]
    }

for vital in spec["vitals"]:
    feature_map[vital["VitalSignID"]] = {
        "category": "vital",
        "fsn": vital["FSN"],
        "sctid": vital["SCTID"]
    }

for risk_factor in spec["risk_factors"]:
    feature_map[risk_factor["RiskFactorID"]] = {
        "category": "risk_factor",
        "fsn": risk_factor["FSN"],
        "sctid": risk_factor["SCTID"]
    }

#### 1. Construction of RDF/Turtle Knowledge Graph

In [4]:
#Initialize empty graph
g = Graph()

#Create namespace for project usign schema.org and SNOMED CT
SCHEMA = Namespace("http://schema.org/")
SNOMED = Namespace("http://snomed.info/id/")
OBO = Namespace("http://purl.obolibrary.org/obo/")
#Create custom namespace for properties not covered by standard vocabulary
PAL = Namespace("http://example.org/opioid-toxicity-palliative/")

#Add namespace prefix for nodes
g.bind("schema", SCHEMA, override=True, replace=True)
g.bind("snomed", SNOMED)
g.bind("pal", PAL)
g.bind("rdf", RDF)
g.bind("rdfs", RDFS)
g.bind("owl", OWL)
g.bind("obo", OBO)

#Declare properties in OWL
g.add((OBO.RO_0002452, RDF.type, OWL.ObjectProperty))
g.add((OBO.RO_0002452, RDFS.label, Literal("has symptom")))
g.add((OBO.RO_0002200, RDF.type, OWL.ObjectProperty))
g.add((OBO.RO_0002200, RDFS.label, Literal("has phenotype")))
g.add((SCHEMA.healthCondition, RDF.type, OWL.ObjectProperty))
g.add((SCHEMA.identifier, RDF.type, OWL.DatatypeProperty))
g.add((PAL.hasOpioidToxicityRisk, RDF.type, OWL.DatatypeProperty))


#Iterate through patients
for index, row in df.iterrows():

    #Define patient node
    patient = PAL[str(row["patient_ID"])]
    g.add((patient, RDF.type, SCHEMA.Patient))
    g.add((patient, SCHEMA.identifier, Literal(str(row["patient_ID"]))))
    #Define patient's risk level
    g.add((patient, PAL.hasOpioidToxicityRisk, Literal(row['Outcome'])))

    #Iterate through all symptoms, vitals and risk factors
    for column in feature_map:
        value = str(row[column])
        #Skip features that are not present in patient condition
        if value == "No" or value == "not_asked":
            continue

        #Get feature type, name and snomed code
        category = feature_map[column]["category"]
        name = feature_map[column]["fsn"]
        snomed_id = feature_map[column]["sctid"]
        
        #Define symptom nodes with snomed codes
        if category == "symptom":
            symptom = SNOMED[snomed_id]
            g.add((symptom, RDF.type, SCHEMA.MedicalSymptom))
            g.add((symptom, RDFS.label, Literal(name)))
            g.add((patient, OBO.RO_0002452, symptom))

        #Define vital nodes with snomed codes
        elif category == "vital":
            vital = SNOMED[snomed_id]
            g.add((vital, RDF.type, SCHEMA.VitalSign))
            g.add((vital, RDFS.label, Literal(name)))
            g.add((patient, OBO.RO_0002200, vital))

        #Define risk factor nodes with snomed codes
        elif category == "risk_factor":
            risk_factor = SNOMED[snomed_id]
            g.add((risk_factor, RDF.type, SCHEMA.MedicalRiskFactor))
            g.add((risk_factor, RDFS.label, Literal(name)))
            g.add((patient, SCHEMA.healthCondition, risk_factor))

#Create and print graph
ttl_content = g.serialize(format="turtle")
rdf_prefix = '@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .\n'
ttl_content = rdf_prefix + ttl_content

with open("knowledge_graph.ttl", "w") as f:
    f.write(ttl_content)

#### 2. Creation of network graph for visualization

In [ ]:
#Create empty dictionary for all mandatory features, excluding optional variables (e.g symptom evolution)
core_features = feature_map

#Initialize empty network graph
G = nx.Graph()

#Iterate through patients and their features
for index, row in df.iterrows():
    #Get patient id and risk level
    patient_id = str(row["patient_ID"])
    risk_level = str(row["Outcome"])

    # Add patient node
    G.add_node(patient_id,label=patient_id,type="Patient",riskClass=risk_level)

    #Skip features that are not present in patient record
    for feature_id in core_features:
        value = str(row[feature_id])
        if value == "No" or value == "not_asked":
            continue

        #Get feature name and category
        feat_name = feature_map[feature_id]["fsn"]
        feat_type = feature_map[feature_id]["category"]

        #Add node for each feature
        G.add_node(feat_name,label=feat_name,type=feat_type)

        #Add edge to connect feature to patient record
        G.add_edge(patient_id,feat_name,label="has_" + feat_type)

#Save graph
nx.write_graphml(G, "knowledge_graph.graphml")